In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetData(trackedArray):
    pIndices=trackedArray[:,0]
    
    varNames=GetVarNames()
    
    dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
    for t in tqdm(range(len(ModelData.timeStrings)), position=2, leave=False):
        for varName in varNames:
            dataDictionary[varName][t] = CallLagrangianArray(ModelData, DataManager, ModelData.timeStrings[t], varName)[pIndices]

    return dataDictionary

def GetVarNames():
    varNames = ['z','LFC','W','QCQI','QV','THETA_v','THETA_e','BUOYANCY2'] #... ['HMC']
    return varNames

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def LimitTrackedArraysRows(trackedArrays,
                           limit=10000): #* use jobarray in the future
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            array = trackedArrays[parcelType][parcelDepth]
            trackedArrays[parcelType][parcelDepth]\
            = trackedArrays[parcelType][parcelDepth][:limit, :]

def ComputeTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(variableData.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

In [ ]:
def RunCode(trackedArrays):

    parcelTypes=['CL','nonCL']
    # parcelDepths=['SHALLOW','DEEP']
    parcelDepths=['SHALLOW','DEEP','ALL'] #*testing
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    for parcelType in tqdm(parcelTypes, position=0, leave=True):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", position=1, leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int); t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData(trackedArray)
            
            #getting data
            varNames=GetVarNames()
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = ComputeTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray

    return trajectoriesArrayDictionary

In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    # trackedArray = MakeTestArray() #*testing
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
    LimitTrackedArraysRows(trackedArrays) #limiting number of parcels to allow computation to complete

    trajectoriesArrayDictionary = RunCode(trackedArrays)
    # TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
    #                                               trajectoriesArrayDictionary, dataName, t='all_times')

In [ ]:
##############################################
#FUNCTIONS (LFC or CB relative)

In [ ]:
# Calculating Functions
def PrepareData(dataDictionary,p=0,varName='W',relative='LFC'):
    #getting
    z = dataDictionary['z'][:,p]
    qcqi = dataDictionary['QCQI'][:,p]
    LFC = dataDictionary['LFC'][:,p]
    data = dataDictionary[varName][:,p]
    time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

    mask = np.isfinite(z)
    
    #masking
    time = time[mask]; time_relative = time-time[0]; z=z[mask]
    data=data[mask]
    if relative=='LFC':
        z_bins = np.arange(-3000, 3000, 50)
        LFC=LFC[mask]; z_relative = z-LFC
    elif relative=='CB':
        z_bins = np.arange(-3000, 3000, 50)
        cloudMask = qcqi[mask] > 1e-6
        if not np.any(cloudMask):
            time_relative[:] = np.nan
            z_relative = np.full_like(z, np.nan)
            data[:] = np.nan
        else: 
            CB = z[cloudMask][0]
            z_relative = z - CB    
    
    # timeRelativeToWhenLFC=False
    # if timeRelativeToWhenLFC:
    #     whenLFC=np.where(z_relative>0)[0][0]
    #     time_relative-=time_relative[whenLFC] #*time relative to time from LFC

    time_bins = np.arange(0, 60, 1)
    return time_relative,z_relative,data, time_bins,z_bins

# #testing
# p=3
# [time_relative,z_relative,data, time_bins,z_bins] = PrepareData(dataDictionary,p,varName='W')
# plt.plot(time_relative,z_relative)
# plt.ylabel('z - z_LFC (km)'); plt.xlabel('time since ascent (mins)')
# plt.axhline(0, color='grey', zorder=-10, linestyle='--',label='LFC')

def BinData(dataDictionary,varName='W',relative='LFC'):    
    _,_,_, time_bins,z_bins = PrepareData(dataDictionary, 0, varName=varName,relative=relative)
    counts_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    weights_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    
    for p in range(dataDictionary['z'].shape[1]):
        time_relative,z_relative,data, _,_ = PrepareData(dataDictionary, p, varName=varName,relative=relative)
        
        c, xedges, yedges = np.histogram2d(time_relative, z_relative, bins=[time_bins, z_bins])
        w, _, _ = np.histogram2d(time_relative, z_relative, bins=[xedges, yedges], weights=data)
        counts_all += c
        weights_all += w
    
    meanData = np.where(counts_all > 0, weights_all / counts_all, np.nan)
    return meanData, time_bins,z_bins

def MakeCalculations(varNames,parcelTypes,parcelDepth='SHALLOW',relative='LFC'):
    meanDataDictionary = {}
    for varName in tqdm(varNames):
        meanDataDictionary[varName] = {}
        for parcelType in parcelTypes:
            dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
            meanData, time_bins, z_bins = BinData(dataDictionary,varName=varName, relative=relative)
            meanDataDictionary[varName][parcelType] = meanData    
    return meanDataDictionary, time_bins,z_bins

In [ ]:
#Plotting Functions
def InterpolateBinnedData(meanData):
    from scipy.interpolate import interp1d        
    # interpolate over empty time bins for each z level
    meanData_filled = meanData.copy()
    for iz in range(meanData.shape[1]):
        row = meanData[:, iz]
        valid = np.isfinite(row)
        if valid.sum() > 1:
            f = interp1d(time_bins[:-1][valid], row[valid], bounds_error=False, fill_value=np.nan)
            meanData_filled[:, iz] = f(time_bins[:-1])
    return meanData_filled

# def PlotBinnedData(meanData,time_bins,z_bins, interpolate=True, cmap='RdBu', symmetric=True, label='',
#                    fig=None,axis=None, vmin_threshold=None, zLabel='z - z_LFC (m)'):
#     if axis is None:
#             fig, axis = plt.subplots(figsize=(12, 5))
#     if interpolate:
#         meanData = InterpolateBinnedData(meanData)
#     if vmin_threshold is not None:
#         meanData = np.where(meanData >= vmin_threshold, meanData, np.nan)
        
#     if symmetric:
#         extend = 'both'
#         vmax = np.nanpercentile(np.abs(meanData), 95)
#         norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
#     else:
#         extend = 'max'
#         if vmin_threshold is not None:
#             vmin = vmin_threshold 
#         else:
#             vmin = np.nanpercentile(meanData, 5)
#         vmax = np.nanpercentile(meanData, 95)
#         norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        
#     im = axis.pcolormesh(time_bins[:-1], z_bins[:-1], meanData.T, cmap=cmap, norm=norm)
#     plt.colorbar(im, ax=axis, label=label, extend=extend)
#     axis.axhline(0, color='grey', linestyle='--')
#     axis.set_xlabel('time since ascent (mins)')
#     axis.set_ylabel(zLabel)
#     axis.legend()


def PlotBinnedData(meanData_1, meanData_2, time_bins, z_bins,
                   interpolate=True, cmap='RdBu', symmetric=True,
                   label='', vmin_threshold=None,
                   zLabel='z - z_LFC (m)',
                   titles=['Panel 1', 'Panel 2']):

    fig, axes = plt.subplots(2, 1, figsize=(8, 10), sharex=True, sharey=True)
    meanDatas = [meanData_1, meanData_2]

    if interpolate:
        meanDatas = [InterpolateBinnedData(meanData) for meanData in meanDatas]

    if vmin_threshold is not None:
        meanDatas = [np.where(meanData >= vmin_threshold, meanData, np.nan)
                     for meanData in meanDatas]

    if symmetric:
        extend = 'both'
        vmax = np.nanmax([np.nanpercentile(np.abs(meanData), 95)
                          for meanData in meanDatas])
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    else:
        extend = 'max'

        if vmin_threshold is not None:
            vmin = vmin_threshold
        else:
            vmin = np.nanmin([np.nanpercentile(meanData, 5)
                              for meanData in meanDatas])

        vmax = np.nanmax([np.nanpercentile(meanData, 95)
                          for meanData in meanDatas])

        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for axis, meanData, title in zip(axes, meanDatas, titles):

        im = axis.pcolormesh(time_bins[:-1], z_bins[:-1],
                             meanData.T, cmap=cmap, norm=norm)

        axis.axhline(0, color='grey', linestyle='--')
        axis.set_ylabel(zLabel)
        axis.set_title(title)

    axes[-1].set_xlabel('time since right before ascent (mins)')
    fig.colorbar(im, ax=axes, label=label, extend=extend)

    return fig

In [ ]:
##############################################
#PLOTTING (LFC Relative Histograms)

In [ ]:
#Making Calculations
varNames=['QV','QCQI','THETA_v','W','BUOYANCY2']
parcelTypes = ['CL', 'nonCL']
[meanDataDictionary, time_bins,z_bins] = MakeCalculations(varNames,parcelTypes,
                                                          parcelDepth='ALL',
                                                          relative='LFC',)

In [ ]:
#Plotting

In [ ]:
for varName in tqdm(varNames):
    vmin_threshold = 1e-6 if varName == "QCQI" else None
    meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
    meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
    fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins, cmap='turbo', symmetric=False, label=varName, zLabel='z - z_LFC (m)',
                   titles=parcelTypes,vmin_threshold=vmin_threshold)

In [ ]:
##############################################
#PLOTTING (CB-relative)

In [ ]:
#Making Calculations
varNames=['QV','QCQI','THETA_v','W','BUOYANCY2']
parcelTypes = ['CL', 'nonCL']
[meanDataDictionary, time_bins,z_bins] = MakeCalculations(varNames,parcelTypes,
                                                          parcelDepth='ALL',
                                                          relative='CB')

In [ ]:
for varName in tqdm(varNames):
    vmin_threshold = 1e-6 if varName == "QCQI" else None
    meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
    meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
    fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins, cmap='turbo', symmetric=False, label=varName, zLabel='z - z_CB(m)',
                   titles=parcelTypes,vmin_threshold=vmin_threshold)

In [ ]:
##############################################
#FUNCTIONS (CB vs CT distribution)

In [ ]:
def CalculateData_CBvsCTDistribution(dataDictionary,varName):
    CB_Values,CE_Z_Values,LFC_Values=[],[],[]
    for p in tqdm(range(dataDictionary['z'].shape[1])):
        [CBIndex,_] = GetTrajectoryCloudBaseTime(dataDictionary,p)
        [LFCIndex,_] = GetTrajectoryLFCTime(dataDictionary,p)
        [exitIndex,_] = GetTrajectoryLastCloudExitTime(dataDictionary,p)
        if CBIndex:
            varAtCB=dataDictionary[varName][:,p][CBIndex].item()
            varAtLFC=dataDictionary[varName][:,p][LFCIndex].item()
            zAtCE=dataDictionary['z'][:,p][exitIndex].item()
            zAtCB=dataDictionary['z'][:,p][CBIndex].item()
            zAtCE_MINUS_zAtCB=zAtCE-zAtCB

            # if zAtCE_MINUS_zAtCB<0: #*testing
            #     print(p)
            
            CB_Values.append(varAtCB)
            LFC_Values.append(varAtLFC)
            CE_Z_Values.append(zAtCE_MINUS_zAtCB)
            
    return CB_Values,LFC_Values,CE_Z_Values

def GetTrajectoryLastCloudExitTime(dataDictionary,p):
    array=dataDictionary['z'][:,p]
    exitIndex=FindCloudExitTime(array)
    exitValue = array[exitIndex]
    return exitIndex,exitValue
def GetTrajectoryCloudBaseTime(dataDictionary,p):
    array=dataDictionary['QCQI'][:,p]
    CBIndex=FindCloudBaseTime(array)
    CBValue = array[CBIndex]
    return CBIndex,CBValue
def GetTrajectoryLFCTime(dataDictionary,p):
    z=dataDictionary['z'][:,p]
    LFC=dataDictionary['LFC'][:,p]
    LFCIndex=FindLFCTime(z,LFC)
    LFCValue = array[LFCIndex]
    return LFCIndex,LFCValue
    
def FindCloudExitTime(array):
    exitIndex = len(array) - 1 - np.argmax(~np.isnan(array[::-1])) - 1
    return exitIndex
def FindCloudBaseTime(array):
    CBIndex = np.where(array > 1e-6)[0]
    if len(CBIndex) == 0:
        return []
    return CBIndex[0]
def FindLFCTime(z,LFC):
    LFCIndex = np.where(z > LFC)[0]
    if len(LFCIndex) == 0:
        return []
    return LFCIndex[0]

In [ ]:
# def MakePlot_CBvsCEDistribution(CB_Values,CE_Z_Values, varName,
#                                 cmap='turbo'):
#     fig, axis = plt.subplots()
#     x, y = CB_Values, CE_Z_Values
#     yLabel = 'z_CE - z_CB'
#     scatterPlot = axis.scatter(x, y, c=y, cmap=cmap)
#     colorbar = fig.colorbar(scatterPlot, ax=axis); colorbar.set_label(yLabel)
#     axis.set_xlabel(f"{varName} at CB"); axis.set_ylabel(yLabel);
#     return fig

def MakePlot_CBvsCEDistribution_2Panel(CB_Values_CL,CE_Z_Values_CL,
                                       CB_Values_nonCL,CE_Z_Values_nonCL, varName,
                                       cmap='turbo'):

    fig, axes = plt.subplots(2, 1,figsize=(7, 8),sharey=True,sharex=True)
    yLabel = 'z_CE - z_CB'

    # --- CL ---
    scatterPlot1 = axes[0].scatter(CB_Values_CL,CE_Z_Values_CL,c=CE_Z_Values_CL,
                                   cmap=cmap)
    colorbar1 = fig.colorbar(scatterPlot1, ax=axes[0]); colorbar1.set_label(yLabel)
    axes[0].set_ylabel(yLabel); axes[0].set_title('CL')

    # --- nonCL ---
    scatterPlot2 = axes[1].scatter(CB_Values_nonCL,CE_Z_Values_nonCL,c=CE_Z_Values_nonCL,
                                   cmap=cmap)
    colorbar2 = fig.colorbar(scatterPlot2, ax=axes[1]); colorbar2.set_label(yLabel)
    axes[1].set_xlabel(f"{varName} at CB"); axes[1].set_ylabel(yLabel); axes[1].set_title('nonCL')

    fig.tight_layout()
    return fig

def MakePlot_CBvsCEDistribution_2DHistogram(CB_Values_CL,CE_Z_Values_CL,
                                       CB_Values_nonCL,CE_Z_Values_nonCL,
                                       varName,cmap='turbo',useLog=False):

    fig, axes = plt.subplots(2, 1,figsize=(7, 8),sharex=True,sharey=True)

    varBinDictionary = GetVarBinDictionary()
    zBins = varBinDictionary['z']
    varBins = varBinDictionary[varName]

    yLabel = 'z_CE - z_CB'

    histCL, xEdges, yEdges = np.histogram2d(CB_Values_CL,CE_Z_Values_CL,bins=[varBins, zBins])
    histNonCL, _, _ = np.histogram2d(CB_Values_nonCL,CE_Z_Values_nonCL,bins=[varBins, zBins])
    norm = LogNorm(vmin=1, vmax=max(np.nanmax(histCL), np.nanmax(histNonCL))) if useLog else None

    plot1 = axes[0].pcolormesh(xEdges,yEdges,histCL.T,cmap=cmap,shading='auto',norm=norm)
    colorbar1 = fig.colorbar(plot1, ax=axes[0]); colorbar1.set_label('Counts')

    axes[0].set_ylabel(yLabel); axes[0].set_title('CL');

    plot2 = axes[1].pcolormesh(xEdges,yEdges,histNonCL.T,cmap=cmap,shading='auto',norm=norm)
    colorbar2 = fig.colorbar(plot2, ax=axes[1]); colorbar2.set_label('Counts')

    axes[1].set_xlabel(f'{varName} at CB'); axes[1].set_ylabel(yLabel); axes[1].set_title('nonCL')

    fig.tight_layout()
    return fig 

def GetVarBinDictionary(numCount=50,zResolution=250):
    varBinDictionary = {'QV':        np.linspace(0, 0.016, numCount),
                        'QCQI':      np.linspace(0, 0.005, numCount),
                        'THETA_v':   np.linspace(305, 335, numCount),
                        'W':         np.linspace(-5, 30, numCount),
                        'BUOYANCY2': np.linspace(-0.1, 0.15, numCount),
                        'z':         np.arange(0, 12000 + zResolution, zResolution)}
    return varBinDictionary

In [ ]:
def MakePlot_SingleLevelDistribution(CB_Values_CL, CB_Values_nonCL, varName, numBins=51,alpha=1,
                            atWhichLevel='CB'):

    fig, axis = plt.subplots()

    axis.hist(CB_Values_CL, bins=numBins, color='blue', alpha=alpha, label='CL')
    axis.hist(CB_Values_nonCL, bins=numBins, color='black', alpha=alpha, label='nonCL')

    axis.set_xlabel(f'{varName} at {atWhichLevel}')
    axis.set_ylabel('Count')
    axis.legend()

    return fig

def MakePlot_CBvsLFCScatter(CB_Values_CL, LFC_Values_CL,
                            CB_Values_nonCL, LFC_Values_nonCL,
                            varName, cmap='turbo'):

    fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True, sharey=True)

    scatter1 = axes[0].scatter(CB_Values_CL, LFC_Values_CL,
                               c=LFC_Values_CL, cmap=cmap)

    colorbar1 = fig.colorbar(scatter1, ax=axes[0])
    colorbar1.set_label(f'{varName} at LFC')

    axes[0].set_ylabel(f'{varName} at LFC')
    axes[0].set_title('CL')

    scatter2 = axes[1].scatter(CB_Values_nonCL, LFC_Values_nonCL,
                               c=LFC_Values_nonCL, cmap=cmap)

    colorbar2 = fig.colorbar(scatter2, ax=axes[1])
    colorbar2.set_label(f'{varName} at LFC')

    axes[1].set_xlabel(f'{varName} at CB')
    axes[1].set_ylabel(f'{varName} at LFC')
    axes[1].set_title('nonCL')

    fig.tight_layout()

    return fig

In [ ]:
##############################################
#PLOTTING (CB vs CT distribution)

In [ ]:
parcelType='SHALLOW'
parcelType='DEEP'
parcelType='ALL'

varNames=['QV','QCQI','THETA_v','W','BUOYANCY2','z']
for varName in varNames:
    #############################
    parcelDepth='CL'
    dataDictionary=trajectoriesArrayDictionary[parcelDepth][parcelType]
    [CB_Values_CL,LFC_Values_CL,CE_Z_Values_CL] = CalculateData_CBvsCTDistribution(dataDictionary,varName)
    
    parcelDepth='nonCL'
    dataDictionary=trajectoriesArrayDictionary[parcelDepth][parcelType]
    [CB_Values_nonCL,LFC_Values_nonCL,CE_Z_Values_nonCL] = CalculateData_CBvsCTDistribution(dataDictionary,varName)

    #Figure One
    fig = MakePlot_CBvsCEDistribution_2Panel(CB_Values_CL,CE_Z_Values_CL,
                                             CB_Values_nonCL,CE_Z_Values_nonCL, varName)
    # #Figure Three
    # MakePlot_SingleLevelDistribution(CB_Values_CL, CB_Values_nonCL, varName, atWhichLevel='CB')
    # #Figure Three
    # MakePlot_SingleLevelDistribution(LFC_Values_CL, LFC_Values_nonCL, varName, atWhichLevel='LFC')

    # #Figure Four
    # MakePlot_CBvsLFCScatter(CB_Values_CL,LFC_Values_CL,
    #                         CB_Values_nonCL,LFC_Values_nonCL, varName)

    # #Figure Five
    # fig = MakePlot_CBvsCEDistribution_2DHistogram(CB_Values_CL,CE_Z_Values_CL,
    #                                          CB_Values_nonCL,CE_Z_Values_nonCL, varName,
    #                                              useLog=True)

In [ ]:
########################################
#TESTING

In [ ]:
#testing parcel trajectory cloudiness & dwdt

p=2
# p=0

parcelType='CL';parcelDepth='SHALLOW'
dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
z=dataDictionary['z']
LFC=dataDictionary['LFC']
qcqi=dataDictionary['QCQI']
w=dataDictionary['W']; 
dwdt = np.full_like(w, np.nan); dwdt[1:] = np.diff(w, axis=0) / ModelData.dt
buoyancy=dataDictionary['BUOYANCY2']
a1=qcqi[:,p]
a2=buoyancy[:,p]
a3=dwdt[:,p]
b=z[:,p]
c=LFC[:,p]

#qcqi
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, gridspec_kw={"wspace":0.03})
axis=axes[0]
axis.plot(a1,b)
axis.set_xlabel('qc+qi (kg/kg)'); axis.set_ylabel('z (m)')
axis.axvline(1e-6,color='gray',linestyle='--',zorder=-10)
apply_scientific_notation([axis])

#w
axis=axes[1]
axis.plot(a2,b)
axis.set_xlabel('B (m/s/s)')#; axis.set_ylabel('z (m)')
axis.axvline(0,color='gray',linestyle='--',zorder=-10)

axis=axes[2]
axis.plot(a3,b)
axis.set_xlabel(r'$\partial w / \partial t$ (m/s/s)')#; axis.set_ylabel('z (m)')
axis.axvline(0,color='gray',linestyle='--',zorder=-10)

#both
for axis in axes:
    axis.axhline(c[(b>c).argmax()],label='lfc',color='orange',linestyle='-',zorder=-5)